# Model 2: Generic Jump-Diffusion

Extends baseline by separating volatility into diffusion and jump components.

**Model:**
```
dA = μA·dt + σ_diff·A·dW + J·A·dN
```

**Approach:**
1. Detect jumps in MSCI returns (|return| > threshold)
2. Estimate jump parameters: λ (intensity), μ_J (mean), σ_J (vol)
3. Compute jump-adjusted volatility
4. Compute d₂ with adjusted vol
5. Compare with baseline

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import norm, linregress
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Paths - uses output from baseline model
PANEL_PATH = "data/processed/cca_baseline_results.csv"

# Model parameters
T = 1.0
JUMP_THRESHOLD = 2.5  # Standard deviations for jump detection

---
## 1. Load Baseline Panel

In [3]:
panel = pd.read_csv(PANEL_PATH, parse_dates=['date'])
print(f"Loaded: {len(panel)} obs, {panel['country'].nunique()} countries")
print(f"Columns: {list(panel.columns)}")
panel.head()

Loaded: 13213 obs, 20 countries
Columns: ['date', 'debt_st', 'debt_lt', 'default_barrier', 'msci_index', 'msci_ret_weekly', 'n_obs', 'msci_vol_annual', 'country', 'cds_spread', 'rf', 'country_lower', 'A', 'd2', 'pd_model', 'spread_model', 'ln_cds', 'd_ln_cds', 'd_pd', 'ym']


,date,debt_st,debt_lt,default_barrier,msci_index,msci_ret_weekly,n_obs,msci_vol_annual,country,cds_spread,rf,country_lower,A,d2,pd_model,spread_model,ln_cds,d_ln_cds,d_pd,ym
0,2010-01-01,0.0,6.957782e+10,3.478891e+10,1317.519,0.026156,5,0.345426,Brazil,122.5600,0.0047,brazil,1.050232e+11,1.074691,0.141257,282.513275,4.808601,NaN,NaN,2010-01
1,2010-01-08,0.0,6.957782e+10,3.478891e+10,1356.236,0.028963,5,0.345288,Brazil,116.0600,0.0037,brazil,1.081094e+11,1.106462,0.134263,268.526790,4.754107,-0.054493,-0.006993,2010-01
2,2010-01-15,0.0,6.957782e+10,3.478891e+10,1307.133,-0.036877,5,0.337814,Brazil,129.0600,0.0033,brazil,1.041953e+11,1.096373,0.136458,272.915698,4.860277,0.106170,0.002194,2010-01
3,2010-01-22,0.0,6.957782e+10,3.478891e+10,1228.794,-0.061803,5,0.343277,Brazil,135.5600,0.0030,brazil,9.795067e+10,0.984334,0.162476,324.951167,4.909414,0.049137,0.026018,2010-01
4,2010-01-29,0.0,6.957782e+10,3.478891e+10,1176.188,-0.043754,5,0.346948,Brazil,143.5629,0.0030,brazil,9.375729e+10,0.909358,0.181581,363.161066,4.966773,0.057359,0.019105,2010-01


In [4]:
# Filter to MSCI EM + date range
msci_em = [
    'brazil', 'chile', 'china', 'colombia', 'czechia', 'egypt', 
    'greece', 'hungary', 'india', 'indonesia', 'south korea',
    'malaysia', 'mexico', 'peru', 'philippines', 'poland', 
    'saudi arabia', 'south africa', 'thailand', 'turkey'
]

panel['country_lower'] = panel['country'].str.strip().str.lower()
panel = panel[panel['country_lower'].isin(msci_em)]
panel = panel[panel['date'] >= '2010-01-01']

print(f"Filtered: {len(panel)} obs, {panel['country'].nunique()} countries")

Filtered: 13213 obs, 20 countries


---
## 2. Compute Weekly Returns

In [5]:
# Compute log returns if not already present
panel = panel.sort_values(['country', 'date'])

if 'msci_ret_weekly' not in panel.columns:
    panel['msci_ret_weekly'] = panel.groupby('country')['msci_index'].transform(
        lambda x: np.log(x / x.shift(1))
    )

panel['msci_ret_weekly'].describe()

count    13213.000000
mean        -0.000215
std          0.037305
min         -0.507798
25%         -0.018656
50%          0.001410
75%          0.020167
max          0.277007
Name: msci_ret_weekly, dtype: float64

---
## 3. Jump Detection

Identify jumps as returns exceeding threshold × rolling standard deviation.

In [6]:
def detect_jumps(group, threshold=2.5, rolling_window=52):
    """
    Detect jumps in return series.
    Jump = |return| > threshold × rolling_std
    """
    df = group.copy()
    
    # Rolling std (backward looking)
    df['rolling_std'] = df['msci_ret_weekly'].rolling(
        window=rolling_window, min_periods=12
    ).std()
    
    # Jump indicator
    df['jump_threshold'] = threshold * df['rolling_std']
    df['is_jump'] = df['msci_ret_weekly'].abs() > df['jump_threshold']
    
    # Jump size (only for jump weeks)
    df['jump_size'] = np.where(df['is_jump'], df['msci_ret_weekly'], np.nan)
    
    return df

# Apply jump detection per country
panel = panel.groupby('country', group_keys=False).apply(
    lambda g: detect_jumps(g, threshold=JUMP_THRESHOLD)
)

# Summary
jump_summary = panel.groupby('country').agg(
    n_obs=('is_jump', 'count'),
    n_jumps=('is_jump', 'sum'),
    jump_rate=('is_jump', 'mean')
).round(4)
jump_summary['jumps_per_year'] = jump_summary['jump_rate'] * 52

print("Jump detection summary:")
print(jump_summary.sort_values('jump_rate', ascending=False))

KeyError: 'country'

In [ ]:
# Visualize jumps for one country
sample_country = 'Turkey'
sample = panel[panel['country'] == sample_country].copy()

fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(sample['date'], sample['msci_ret_weekly'], alpha=0.7, label='Returns')
ax.fill_between(sample['date'], -sample['jump_threshold'], sample['jump_threshold'], 
                alpha=0.2, color='gray', label=f'±{JUMP_THRESHOLD}σ threshold')

# Mark jumps
jumps = sample[sample['is_jump']]
ax.scatter(jumps['date'], jumps['msci_ret_weekly'], color='red', s=30, 
           label=f'Jumps (n={len(jumps)})', zorder=5)

ax.set_xlabel('Date')
ax.set_ylabel('Weekly Return')
ax.set_title(f'{sample_country}: Jump Detection (threshold = {JUMP_THRESHOLD}σ)')
ax.legend()
ax.axhline(0, color='k', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

---
## 4. Estimate Jump Parameters by Country

In [ ]:
def estimate_jump_params(group):
    """
    Estimate jump-diffusion parameters for a country.
    
    Returns:
        lambda_: Jump intensity (jumps per year)
        mu_J: Mean jump size
        sigma_J: Jump size volatility
        sigma_diff: Diffusion volatility (from non-jump returns)
    """
    df = group.dropna(subset=['msci_ret_weekly', 'is_jump'])
    
    # Separate jump and non-jump returns
    jump_returns = df[df['is_jump']]['msci_ret_weekly']
    nonjump_returns = df[~df['is_jump']]['msci_ret_weekly']
    
    # Jump parameters
    n_jumps = len(jump_returns)
    n_total = len(df)
    
    lambda_ = (n_jumps / n_total) * 52  # Annualized intensity
    
    if n_jumps > 0:
        mu_J = jump_returns.mean()
        sigma_J = jump_returns.std() if n_jumps > 1 else abs(jump_returns.mean())
    else:
        mu_J = 0
        sigma_J = 0
    
    # Diffusion volatility (from non-jump returns, annualized)
    sigma_diff = nonjump_returns.std() * np.sqrt(52) if len(nonjump_returns) > 1 else 0.2
    
    return pd.Series({
        'lambda': lambda_,
        'mu_J': mu_J,
        'sigma_J': sigma_J,
        'sigma_diff': sigma_diff,
        'n_jumps': n_jumps,
        'n_obs': n_total
    })

# Estimate for each country
jump_params = panel.groupby('country').apply(estimate_jump_params)
print("Jump parameters by country:")
print(jump_params.round(4))

In [ ]:
# Visualize jump parameters
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Lambda (jump intensity)
ax = axes[0]
jump_params_sorted = jump_params.sort_values('lambda', ascending=True)
ax.barh(jump_params_sorted.index, jump_params_sorted['lambda'])
ax.set_xlabel('λ (jumps per year)')
ax.set_title('Jump Intensity')

# Mean jump size
ax = axes[1]
ax.barh(jump_params_sorted.index, jump_params_sorted['mu_J'])
ax.axvline(0, color='k', linestyle='--', alpha=0.3)
ax.set_xlabel('μ_J (mean jump size)')
ax.set_title('Mean Jump Size')

# Diffusion vs Total vol
ax = axes[2]
ax.barh(jump_params_sorted.index, jump_params_sorted['sigma_diff'], label='σ_diff')
ax.set_xlabel('Volatility (annualized)')
ax.set_title('Diffusion Volatility')

plt.tight_layout()
plt.show()

---
## 5. Compute Jump-Adjusted Volatility and d₂

Merton (1976) jump-diffusion total variance:
```
σ²_total = σ²_diff + λ × (μ_J² + σ_J²)
```

In [ ]:
# Merge jump parameters into panel
panel = panel.merge(
    jump_params[['lambda', 'mu_J', 'sigma_J', 'sigma_diff']], 
    left_on='country', 
    right_index=True,
    how='left',
    suffixes=('', '_param')
)

print(f"Panel columns: {list(panel.columns)}")

In [ ]:
# Compute jump-adjusted volatility
# Option A: Use country-level parameters (constant)
panel['sigma_jump_adj'] = np.sqrt(
    panel['sigma_diff']**2 + 
    panel['lambda'] * (panel['mu_J']**2 + panel['sigma_J']**2)
)

# Compare with baseline volatility
print("Volatility comparison:")
vol_compare = panel.groupby('country').agg(
    baseline_vol=('msci_vol_annual', 'mean'),
    diffusion_vol=('sigma_diff', 'first'),
    jump_adj_vol=('sigma_jump_adj', 'first')
).round(4)
vol_compare['diff_vs_baseline'] = vol_compare['jump_adj_vol'] - vol_compare['baseline_vol']
print(vol_compare)

In [ ]:
# Compute d₂ with jump-adjusted volatility
def compute_d2(A, B, sigma, r, T=1.0):
    with np.errstate(divide='ignore', invalid='ignore'):
        d2 = (np.log(A / B) + (r - sigma**2 / 2) * T) / (sigma * np.sqrt(T))
    return d2

# d₂ with jump-adjusted vol
panel['d2_jump'] = compute_d2(
    A=panel['A'],
    B=panel['default_barrier'],
    sigma=panel['sigma_jump_adj'],
    r=panel['rf'],
    T=T
)

# PD with jump model
panel['pd_jump'] = norm.cdf(-panel['d2_jump'])

# Compare d₂ distributions
print("d₂ comparison:")
print(f"Baseline d₂: mean={panel['d2'].mean():.3f}, std={panel['d2'].std():.3f}")
print(f"Jump d₂:     mean={panel['d2_jump'].mean():.3f}, std={panel['d2_jump'].std():.3f}")

---
## 6. Regression: Compare Models

In [ ]:
# Compute changes
panel = panel.sort_values(['country', 'date'])
panel['ln_cds'] = np.log(panel['cds_spread'])
panel['d_ln_cds'] = panel.groupby('country')['ln_cds'].diff()
panel['d_pd_baseline'] = panel.groupby('country')['pd_model'].diff()
panel['d_pd_jump'] = panel.groupby('country')['pd_jump'].diff()

# Time FE
panel['ym'] = panel['date'].dt.to_period('M').astype(str)

# Clean data
reg_data = panel.dropna(subset=['d_ln_cds', 'd_pd_baseline', 'd_pd_jump'])
reg_data = reg_data.replace([np.inf, -np.inf], np.nan).dropna(subset=['d_ln_cds', 'd_pd_baseline', 'd_pd_jump'])

print(f"Regression sample: {len(reg_data)} obs")

In [ ]:
# Model 1: Baseline
model_baseline = smf.ols('d_ln_cds ~ d_pd_baseline + C(country) + C(ym)', data=reg_data).fit()

# Model 2: Jump-adjusted
model_jump = smf.ols('d_ln_cds ~ d_pd_jump + C(country) + C(ym)', data=reg_data).fit()

print("="*60)
print("MODEL COMPARISON")
print("="*60)
print(f"{'':20} {'Baseline':>15} {'Jump-Adjusted':>15}")
print("-"*60)
print(f"{'β (ΔPD)':20} {model_baseline.params['d_pd_baseline']:>15.4f} {model_jump.params['d_pd_jump']:>15.4f}")
print(f"{'t-stat':20} {model_baseline.tvalues['d_pd_baseline']:>15.2f} {model_jump.tvalues['d_pd_jump']:>15.2f}")
print(f"{'R²':20} {model_baseline.rsquared:>15.4f} {model_jump.rsquared:>15.4f}")
print("-"*60)
print(f"{'R² improvement':20} {(model_jump.rsquared - model_baseline.rsquared):>15.4f}")
print(f"{'% improvement':20} {(model_jump.rsquared / model_baseline.rsquared - 1)*100:>14.2f}%")

---
## 7. Split: Oil Exporters vs Controls

In [ ]:
# Define groups
oil_exporters = ['saudi arabia', 'colombia', 'mexico', 'brazil', 'egypt', 'malaysia']
controls = ['indonesia', 'philippines', 'turkey', 'chile', 'china', 
            'south africa', 'south korea', 'thailand']

reg_data['group'] = reg_data['country_lower'].apply(
    lambda x: 'oil_exporter' if x in oil_exporters 
              else ('control' if x in controls else 'other')
)

# Filter to oil and control only
reg_data_groups = reg_data[reg_data['group'].isin(['oil_exporter', 'control'])]
print(f"Oil exporters: {reg_data_groups[reg_data_groups['group']=='oil_exporter']['country'].nunique()} countries")
print(f"Controls: {reg_data_groups[reg_data_groups['group']=='control']['country'].nunique()} countries")

In [ ]:
# Run regressions by group
results = []

for group in ['oil_exporter', 'control']:
    sample = reg_data_groups[reg_data_groups['group'] == group]
    
    # Baseline
    m1 = smf.ols('d_ln_cds ~ d_pd_baseline + C(country) + C(ym)', data=sample).fit()
    
    # Jump
    m2 = smf.ols('d_ln_cds ~ d_pd_jump + C(country) + C(ym)', data=sample).fit()
    
    results.append({
        'group': group,
        'n_obs': len(sample),
        'n_countries': sample['country'].nunique(),
        'beta_baseline': m1.params['d_pd_baseline'],
        'r2_baseline': m1.rsquared,
        'beta_jump': m2.params['d_pd_jump'],
        'r2_jump': m2.rsquared,
        'r2_improvement': m2.rsquared - m1.rsquared,
        'r2_pct_improvement': (m2.rsquared / m1.rsquared - 1) * 100
    })

results_df = pd.DataFrame(results)
print("\n" + "="*80)
print("COMPARISON: BASELINE vs GENERIC JUMP MODEL")
print("="*80)
print(results_df.to_string(index=False))

In [ ]:
# Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# R² comparison
ax = axes[0]
x = np.arange(len(results_df))
width = 0.35
ax.bar(x - width/2, results_df['r2_baseline'], width, label='Baseline', alpha=0.8)
ax.bar(x + width/2, results_df['r2_jump'], width, label='Generic Jump', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(results_df['group'])
ax.set_ylabel('R²')
ax.set_title('R² Comparison by Group')
ax.legend()

# R² improvement
ax = axes[1]
colors = ['green' if x > 0 else 'red' for x in results_df['r2_improvement']]
ax.bar(results_df['group'], results_df['r2_improvement'], color=colors, alpha=0.8)
ax.axhline(0, color='k', linestyle='--', alpha=0.3)
ax.set_ylabel('R² Improvement')
ax.set_title('R² Improvement (Jump - Baseline)')

# Beta comparison
ax = axes[2]
ax.bar(x - width/2, results_df['beta_baseline'], width, label='Baseline', alpha=0.8)
ax.bar(x + width/2, results_df['beta_jump'], width, label='Generic Jump', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(results_df['group'])
ax.set_ylabel('β (ΔPD)')
ax.set_title('β Comparison by Group')
ax.legend()

plt.tight_layout()
plt.show()

---
## 8. Country-Level R² Comparison

In [ ]:
# R² by country for both models
country_results = []

for country in reg_data['country'].unique():
    sample = reg_data[reg_data['country'] == country]
    if len(sample) < 50:
        continue
    
    # Simple OLS (no FE needed for single country)
    s1, _, r1, _, _ = linregress(sample['d_pd_baseline'].fillna(0), sample['d_ln_cds'].fillna(0))
    s2, _, r2, _, _ = linregress(sample['d_pd_jump'].fillna(0), sample['d_ln_cds'].fillna(0))
    
    country_lower = country.lower().strip()
    group = 'oil_exporter' if country_lower in oil_exporters else (
            'control' if country_lower in controls else 'other')
    
    country_results.append({
        'country': country,
        'group': group,
        'r2_baseline': r1**2,
        'r2_jump': r2**2,
        'r2_improvement': r2**2 - r1**2
    })

country_df = pd.DataFrame(country_results).sort_values('r2_improvement', ascending=False)
print("R² by country:")
print(country_df.to_string(index=False))

In [ ]:
# Plot by country with group coloring
fig, ax = plt.subplots(figsize=(12, 8))

country_df_sorted = country_df.sort_values('r2_improvement', ascending=True)
colors = ['orange' if g == 'oil_exporter' else ('blue' if g == 'control' else 'gray') 
          for g in country_df_sorted['group']]

ax.barh(country_df_sorted['country'], country_df_sorted['r2_improvement'], color=colors, alpha=0.8)
ax.axvline(0, color='k', linestyle='-', linewidth=0.5)
ax.set_xlabel('R² Improvement (Jump - Baseline)')
ax.set_title('R² Improvement by Country\n(Orange = Oil Exporter, Blue = Control)')

# Add legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='orange', alpha=0.8, label='Oil Exporter'),
    Patch(facecolor='blue', alpha=0.8, label='Control')
]
ax.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.show()

---
## 9. Summary

In [ ]:
print("\n" + "="*80)
print("SUMMARY: GENERIC JUMP MODEL vs BASELINE")
print("="*80)

print("\n1. OVERALL:")
print(f"   Baseline R²: {model_baseline.rsquared:.4f}")
print(f"   Jump R²:     {model_jump.rsquared:.4f}")
print(f"   Improvement: {(model_jump.rsquared - model_baseline.rsquared):.4f}")

print("\n2. BY GROUP:")
for _, row in results_df.iterrows():
    print(f"   {row['group']:15} | Baseline R²: {row['r2_baseline']:.4f} | Jump R²: {row['r2_jump']:.4f} | Δ: {row['r2_improvement']:.4f}")

print("\n3. KEY FINDING:")
oil_imp = results_df[results_df['group']=='oil_exporter']['r2_improvement'].values[0]
ctrl_imp = results_df[results_df['group']=='control']['r2_improvement'].values[0]
print(f"   Oil exporters improvement: {oil_imp:.4f}")
print(f"   Control improvement:       {ctrl_imp:.4f}")
print(f"   Difference-in-differences: {oil_imp - ctrl_imp:.4f}")

In [ ]:
# Save results
panel.to_csv('data/processed/cca_jump_results.csv', index=False)
print("Saved to data/processed/cca_jump_results.csv")